In [ ]:
# CELL 1: Install dependencies
# Run this every time you start/restart the Colab session
# Note: The cells must be run in an order (1-5)

!pip install fastapi uvicorn pyngrok nest_asyncio -q
print("All dependencies installed.")

All dependencies installed.


In [ ]:
# CELL 2: Mount Google Drive and load model

import joblib
from google.colab import drive

drive.mount('/content/drive')

model_path = "/content/drive/My Drive/Spam-Ham Dataset/naive_bayes_model.pkl"
model = joblib.load(model_path)

test_pred = model.predict_proba(["Congratulations! You won a free prize. Click here now."])

print("Model loaded successfully.")
print(f"Sanity check — spam probability: {test_pred[0][1]:.4f} (should be high)")

Mounted at /content/drive
Model loaded successfully.
Sanity check — spam probability: 0.8968 (should be high)


In [ ]:
# CELL 3: Define the FastAPI app and prediction endpoint

from fastapi import FastAPI
from pydantic import BaseModel
import re

app = FastAPI(title="Spam Detection API")

class EmailInput(BaseModel):
    text: str

def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

@app.post("/predict")
def predict(email: EmailInput):
    cleaned = clean_text(email.text)
    proba = model.predict_proba([cleaned])[0]
    spam_prob = float(proba[1])
    label = "SPAM" if spam_prob >= 0.5 else "HAM"
    return {
        "prediction": label,        # "SPAM" or "HAM"
        "confidence": round(spam_prob, 4),
        "flagged": spam_prob >= 0.5
    }

@app.get("/health")
def health():
    return {"status": "ok", "model": "naive_bayes_tfidf"}

print("FastAPI app defined. Proceed to next cell to start the server.")

FastAPI app defined. Proceed to next cell to start the server.


In [ ]:
# CELL 4: Authenticate ngrok, start server + tunnel

import uvicorn
import nest_asyncio
import threading
import time
import socket
from pyngrok import ngrok, conf

nest_asyncio.apply()

NGROK_AUTH_TOKEN = "REPLACE_WITH_YOUR_NGROK_AUTH_TOKEN"

conf.get_default().auth_token = NGROK_AUTH_TOKEN

ngrok.kill()

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
port_in_use = sock.connect_ex(('localhost', 8000)) == 0
sock.close()

if not port_in_use:
    def run_server():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("Uvicorn started.")
else:
    print("Uvicorn already running on port 8000, reusing it.")

tunnel = ngrok.connect(8000)
base_url = tunnel.public_url

print("=" * 55)
print("  SERVER IS LIVE")
print("=" * 55)
print(f"  Health check : {base_url}/health")
print(f"  Predict      : {base_url}/predict")
print(f"  API docs     : {base_url}/docs")
print("=" * 55)
print("  Open /health in your browser to confirm.")
print("  Paste /predict URL into n8n.")
print("=" * 55)

Uvicorn started.
  SERVER IS LIVE
  Health check : https://boxcar-reptile-ladies.ngrok-free.dev/health
  Predict      : https://boxcar-reptile-ladies.ngrok-free.dev/predict
  API docs     : https://boxcar-reptile-ladies.ngrok-free.dev/docs
  Open /health in your browser to confirm.
  Paste /predict URL into n8n.


In [ ]:
# CELL 5 (optional): Test the API locally before using n8n
# Run this to confirm everything works before setting up n8n

import requests

test_cases = [
    {
        "label": "discount spam",
        "text": "LIMITED TIME OFFER! Get 80% off all medications. Order now and save big!"
    },
    {
        "label": "lottery scam",
        "text": "You are the lucky winner of $500,000. Reply immediately to claim your funds."
    },
    {
        "label": "crypto scam",
        "text": "Double your Bitcoin investment in 24 hours. Guaranteed returns. Join today."
    },
    {
        "label": "urgent phishing",
        "text": "Your online banking account has been locked. Click the link below to restore access."
    },
    {
        "label": "fake tech support",
        "text": "Warning: Your computer is infected with multiple viruses. Call our support team immediately."
    },
    {
        "label": "job scam",
        "text": "Work from home and earn $5000 per week with no experience required."
    },
    {
        "label": "meeting follow up",
        "text": "Thank you for attending today's meeting. The presentation slides are attached for reference."
    },
    {
        "label": "project update",
        "text": "The development team has completed the requested changes and deployed them to staging."
    },
    {
        "label": "university email",
        "text": "Please remember to submit your assignment before Friday. Late submissions will incur penalties."
    },
    {
        "label": "calendar invitation",
        "text": "You have been invited to the weekly project review scheduled for Wednesday at 10:00 AM."
    },
    {
        "label": "borderline marketing",
        "text": "We are excited to share our latest product launch. Register now to receive early access."
    },
    {
        "label": "newsletter",
        "text": "Welcome to our monthly newsletter. Here are the latest updates and industry insights."
    }
]

print("Running test predictions against local server...\n")
for case in test_cases:
    response = requests.post(
        "http://localhost:8000/predict",
        json={"text": case["text"]}
    )
    result = response.json()
    flag = "SPAM" if result["flagged"] else " HAM"
    print(f"[{flag}]  confidence={result['confidence']:.4f}  | {case['label']}")

print("\nIf results look sensible, proceed to n8n setup.")

Running test predictions against local server...

[SPAM]  confidence=0.9030  | discount spam
[SPAM]  confidence=0.9408  | lottery scam
[SPAM]  confidence=0.8516  | crypto scam
[SPAM]  confidence=0.7682  | urgent phishing
[SPAM]  confidence=0.5599  | fake tech support
[SPAM]  confidence=0.7412  | job scam
[ HAM]  confidence=0.0823  | meeting follow up
[ HAM]  confidence=0.2079  | project update
[ HAM]  confidence=0.1374  | university email
[ HAM]  confidence=0.0165  | calendar invitation
[SPAM]  confidence=0.6646  | borderline marketing
[SPAM]  confidence=0.7650  | newsletter

If results look sensible, proceed to n8n setup.
